In [81]:
# Using JupyterNotebook (without hints or auto-completes)
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

# Fully Connected Network

In [2]:
# Let's warm up a bit with fully connected network here
class FullyConnectedNetwork(torch.nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        self.gelu = torch.nn.GELU()
        self.fc1 = torch.nn.Linear(input_size, hidden_size)
        self.fc2 = torch.nn.Linear(hidden_size, hidden_size)
        self.fc3 = torch.nn.Linear(hidden_size, output_size)

    def forward(self, x):
        x = self.fc1(x)
        x = self.gelu(x)
        x = self.fc2(x)
        x = self.gelu(x)
        x = self.fc3(x)
        return x

In [3]:
input_size = 10
hidden_size = 4
output_size = 5

In [4]:
fcc = FullyConnectedNetwork(input_size, hidden_size, output_size)

In [5]:
X_numpy_1 = np.random.randn(10, 10, 10, input_size)
X_1 = torch.from_numpy(X_numpy_1).float()

# Try a different-sized input, as long as the last dimension matches, should be fine
X_numpy_2 = np.random.randn(20, input_size)
X_2 = torch.from_numpy(X_numpy_2).float()

X_numpy_3 = np.random.randn(input_size)
X_3 = torch.from_numpy(X_numpy_3).float()

In [6]:
with torch.no_grad():
    output_1 = fcc(X_1)
    output_2 = fcc(X_2)
    output_3 = fcc(X_3)

In [7]:
print(output_1.size())
print(output_2.size())
print(output_3.size())

torch.Size([10, 10, 10, 5])
torch.Size([20, 5])
torch.Size([5])


In [8]:
output_1.requires_grad

False

In [9]:
output_1_numpy = output_1.numpy()
print(output_1_numpy.shape)

(10, 10, 10, 5)


## Train FC

In [10]:
input_size = 7
output_size = 2
hidden_size = 10

In [11]:
fcc = FullyConnectedNetwork(input_size, hidden_size, output_size)

In [12]:
num_epoch = 100
batch_size = 32
learning_rate = 1e-3

In [13]:
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(fcc.parameters(), lr = learning_rate)

In [14]:
X_numpy = np.random.randn(500, input_size)
y_numpy = np.random.randint(0, output_size, (500,))

X = torch.from_numpy(X_numpy).float()
y = torch.from_numpy(y_numpy).long()

In [15]:
dataset = torch.utils.data.TensorDataset(X, y)
dataloader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=True)

In [16]:
report_per_epoch = 10
for epoch in range(num_epoch):
    running_loss = 0.0
    for inputs, labels in dataloader:
        optimizer.zero_grad()
        loss = criterion(fcc(inputs), labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    if (epoch+1) % report_per_epoch == 0:
        print(f"Epoch [{epoch+1}/{num_epoch}], Loss: {running_loss/len(dataloader):.4f}")

print(f"Training Finished!")
        

Epoch [10/100], Loss: 0.6854
Epoch [20/100], Loss: 0.6798
Epoch [30/100], Loss: 0.6760
Epoch [40/100], Loss: 0.6702
Epoch [50/100], Loss: 0.6673
Epoch [60/100], Loss: 0.6637
Epoch [70/100], Loss: 0.6570
Epoch [80/100], Loss: 0.6509
Epoch [90/100], Loss: 0.6446
Epoch [100/100], Loss: 0.6407
Training Finished!


In [17]:
len(dataloader)

16

In [18]:
print(inputs.size())

torch.Size([20, 7])


In [19]:
print(labels.size())

torch.Size([20])


In [20]:
for inputs, labels in dataloader:
    print("inputs.size(): ", inputs.size())
    print("labels.size(): ", labels.size())
    break

inputs.size():  torch.Size([32, 7])
labels.size():  torch.Size([32])


In [21]:
test_input_numpy = np.random.randn(25, input_size)
test_input = torch.from_numpy(test_input_numpy).float()
with torch.no_grad():
    test_logits = fcc(test_input)
    output_binary = torch.argmax(test_logits, dim=1)

In [22]:
output_binary

tensor([0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1,
        0])

In [30]:
print("prob of class 0: ")
print(torch.softmax(test_logits, dim=1)[:,0])
print("prob of class 1: ")
print(torch.softmax(test_logits, dim=1)[:,1])

prob of class 0: 
tensor([0.5522, 0.5636, 0.3845, 0.3265, 0.4101, 0.5111, 0.5239, 0.7175, 0.5551,
        0.5162, 0.4691, 0.4787, 0.3026, 0.2957, 0.4716, 0.5859, 0.4707, 0.3657,
        0.4264, 0.3608, 0.4329, 0.3559, 0.4786, 0.3837, 0.5926])
prob of class 1: 
tensor([0.4478, 0.4364, 0.6155, 0.6735, 0.5899, 0.4889, 0.4761, 0.2825, 0.4449,
        0.4838, 0.5309, 0.5213, 0.6974, 0.7043, 0.5284, 0.4141, 0.5293, 0.6343,
        0.5736, 0.6392, 0.5671, 0.6441, 0.5214, 0.6163, 0.4074])


In [35]:
loss.size()

torch.Size([])

# Single-Head Attention

In [71]:
class SingleHeadAttention(torch.nn.Module):
    def __init__(self, dim_qk, dim_v, dim_embedding):
        super().__init__()
        self.d_qk = dim_qk
        self.w_k = torch.nn.Linear(dim_embedding, dim_qk)
        self.w_q = torch.nn.Linear(dim_embedding, dim_qk)
        self.w_v = torch.nn.Linear(dim_embedding, dim_v)
        # attention does not have layer norm

    def forward(self, x):
        key = self.w_k(x)
        value = self.w_v(x)
        query = self.w_q(x)
        atten = torch.softmax((query @ key.T) / self.d_qk**0.5, dim=-1)
        # transpose is not generic, especially for high-dim inputs
        return atten @ value

In [72]:
# n_head = 8
embedding_dim = 10
qk_dim = 5
v_dim = 6

In [73]:
x_numpy = np.random.randn(25, 10)

In [74]:
x = torch.from_numpy(x_numpy).float()

In [75]:
x.size()

torch.Size([25, 10])

In [76]:
sha = SingleHeadAttention(qk_dim, v_dim, embedding_dim)

In [77]:
with torch.no_grad():
    output = sha(x)

In [78]:
output.size()

torch.Size([25, 6])

In [79]:
output_numpy = output.numpy()

In [70]:
output_numpy.shape

(25, 6)

# Multi-Head Attention (minimum)

We exlude the following for simplicity:  
- attention mask
- causal mask
- dropout

In [108]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.scale = self.d_head**-0.5

        # If stored together, dim will be (d_model, 3*d_model)
        self.w_k = nn.Linear(d_model, d_model, bias=False) # d_model = n_heads * d_head
        self.w_q = nn.Linear(d_model, d_model, bias=False) # d_model = n_heads * d_head
        self.w_v = nn.Linear(d_model, d_model, bias=False) # d_model = n_heads * d_head
        self.output_proj = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x):
        # x.shape = (batch_size, seq_len, d_model)
        batch_size, seq_len, _ = x.shape
        k = self.w_k(x).reshape(batch_size, seq_len, self.n_heads, self.d_head).transpose(1, 2).contiguous()
        q = self.w_q(x).reshape(batch_size, seq_len, self.n_heads, self.d_head).transpose(1, 2).contiguous()
        v = self.w_v(x).reshape(batch_size, seq_len, self.n_heads, self.d_head).transpose(1, 2).contiguous()
        # shape = (batch_size, n_head, seq_len, d_head)
        
        atten = torch.softmax(q @ k.transpose(-2, -1)*self.scale, dim=-1)
        # shape = (batch_size, n_head, seq_len, seq_len)
        
        output = atten @ v
        # shape = (batch_size, n_head, seq_len, d_head)

        output = output.transpose(1, 2).reshape(batch_size, seq_len, self.d_model).contiguous()
        return self.output_proj(output)
        

In [109]:
d_model = 48 # embedding dimension
n_heads = 8
batch_size = 32
seq_len = 25

In [110]:
mha = MultiHeadAttention(d_model, n_heads)

In [111]:
x_numpy = np.random.randn(batch_size, seq_len, d_model)
x = torch.from_numpy(x_numpy).float()

In [114]:
with torch.no_grad():
    output = mha(x)

In [115]:
output.size()

torch.Size([32, 25, 48])

In [116]:
output_numpy = output.numpy()

In [117]:
output_numpy.shape

(32, 25, 48)

# Transformer

In [120]:
# With customized MHA Module
# We do not implement dropout for simplicity

class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff=2048):
        super().__init__()
        self.mha = MultiHeadAttention(d_model, n_heads)
        self.norm1 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model)
        )
        self.norm2 = nn.LayerNorm(d_model)
        
    
    def forward(self, x):
        # We want a residual connection
        x = self.norm1(x)
        x = x + self.mha(x)
        x = self.norm2(x)
        x = x + self.ffn(x)
        return x

In [165]:
class TransformerEncoder(nn.Module):
    def __init__(self, n_layers, d_model, n_heads, d_ff=2048):
        super().__init__()
        self.layers = nn.ModuleList([
            TransformerBlock(d_model, n_heads, d_ff)
            for _ in range(n_layers)
        ])
        self.final_norm = nn.LayerNorm(d_model)
        self.ffn = nn.Linear(d_model, 2)
    
    def forward(self, x):
        for layer in self.layers:
            x = layer(x)
        x = self.final_norm(x)
        # For the binary classification task
        x = x.mean(dim=1) # mean pooling to get the aggregation of entire sequence
        x = self.ffn(x)
        return x

In [166]:
encoder = TransformerEncoder(n_layers=2, d_model=48, n_heads=8)

In [167]:
x_numpy = np.random.randn(1, 20, 48)

In [168]:
x = torch.from_numpy(x_numpy).float()

In [169]:
with torch.no_grad():
    output = encoder(x)

In [170]:
output.requires_grad

False

In [171]:
ouput_numpy = output.numpy()

In [172]:
ouput_numpy.shape

(1, 2)

## Training loop for encoder  
Not a real training without causal mask

In [173]:
num_epochs = 100
learning_rate = 1e-3
batch_size = 32

In [174]:
sample_size = 500

# Binary classification for positive and negative sentiment of a sentence
# dim = (sample_size, seq_len, embed_dim)
x_numpy = np.random.randn(sample_size, 20, 48)
y_numpy = np.random.randint(0, 2, (sample_size,))
x = torch.from_numpy(x_numpy).float()
y = torch.from_numpy(y_numpy).long()

In [175]:
y.size()

torch.Size([500])

In [176]:
optimizer = torch.optim.Adam(encoder.parameters(), lr = learning_rate)
criterion = torch.nn.CrossEntropyLoss()

In [177]:
dataset = torch.utils.data.TensorDataset(x, y)
dataloader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=True)

In [180]:
for epoch in range(num_epochs):
    running_loss = 0.0

    for inputs, labels in dataloader:
        optimizer.zero_grad()
        outputs = encoder(inputs)
        loss = criterion(outputs, labels)
        
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    if (epoch+1) % 10 == 0:
        print(f"Epoch {epoch+1}/{num_epochs}, Loss: {running_loss/len(dataloader):.4f}")

print('Training Finished!')

Epoch 10/100, Loss: 0.2528
Epoch 20/100, Loss: 0.0004
Epoch 30/100, Loss: 0.0002
Epoch 40/100, Loss: 0.0001
Epoch 50/100, Loss: 0.0001
Epoch 60/100, Loss: 0.0001
Epoch 70/100, Loss: 0.0000
Epoch 80/100, Loss: 0.0000
Epoch 90/100, Loss: 0.0000
Epoch 100/100, Loss: 0.0000
Training Finished!


In [195]:
x_test_numpy = np.random.randn(10, 20, 48)
x_test = torch.from_numpy(x_test_numpy).float()
with torch.no_grad():
    y_pred_logits = encoder(x_test)
    output_binary = torch.argmax(y_pred_logits, dim=-1)

In [197]:
print("Prob of class 0:")
torch.softmax(y_pred_logits, dim=-1)[:,0]

Prob of class 0:


tensor([9.9997e-01, 1.3738e-05, 9.8651e-01, 2.2011e-03, 6.9092e-04, 9.9893e-01,
        3.5893e-05, 9.9902e-01, 9.9978e-01, 9.9998e-01])

In [198]:
print("Prob of class 1:")
torch.softmax(y_pred_logits, dim=-1)[:,1]

Prob of class 1:


tensor([3.2897e-05, 9.9999e-01, 1.3488e-02, 9.9780e-01, 9.9931e-01, 1.0673e-03,
        9.9996e-01, 9.8426e-04, 2.2270e-04, 1.6437e-05])

In [194]:
output_binary

tensor([0, 0, 1, 1, 1, 0, 0, 0, 1, 1])

# MHA with Causal Mask

In [233]:
# need to add an output projection, do it better next time
class SelfAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        assert d_model % n_heads == 0, "d_model should be int multiples of n_heads!"
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.scale = self.d_head**-0.5
        
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.W_q = nn.Linear(d_model, d_model, bias=False)

        
    
    def forward(self, x, is_causal=False):
        # dim = (batch_size, seq_len, d_model)
        batch_size, seq_len, _ = x.shape

        # dim = (batch_size, n_heads, seq_len, d_head)
        k = self.W_k(x).reshape(batch_size, seq_len, self.n_heads, self.d_head).transpose(1, 2).contiguous()
        v = self.W_v(x).reshape(batch_size, seq_len, self.n_heads, self.d_head).transpose(1, 2).contiguous()
        q = self.W_q(x).reshape(batch_size, seq_len, self.n_heads, self.d_head).transpose(1, 2).contiguous()

        # dim = (batch_size, n_heads, seq_len, seq_len)
        attn = q @ k.transpose(-2, -1) * self.scale
        
        if is_causal:
            causal_mask = torch.triu(torch.ones(seq_len, seq_len, dtype=bool), diagonal=1)
            attn = attn.masked_fill(causal_mask, -torch.inf)
        
        norm_attn = torch.softmax(attn, dim=-1)

        # dim = (batch_size, n_heads, seq_len, d_head)
        output = (norm_attn @ v).transpose(1, 2).reshape(batch_size, seq_len, self.d_model).contiguous()
        
        return output

In [234]:
d_model = 48
n_heads = 8
sa = SelfAttention(d_model, n_heads)

In [235]:
x_numpy = np.random.randn(1, 20, 48)
x = torch.from_numpy(x_numpy).float()

In [236]:
with torch.no_grad():
    output = sa(x, is_causal=True)

In [237]:
output.size()

torch.Size([1, 20, 48])

In [259]:
# need to add an output projection, do it better next time
class CrossAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        assert d_model % n_heads == 0, "d_model should be int multiples of n_heads!"
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.scale = self.d_head**-0.5

        # Try another implementation, not with nn.Linear but with nn.Parameter
        self.W_k = nn.Parameter(torch.rand(d_model, d_model))
        self.W_q = nn.Parameter(torch.rand(d_model, d_model))
        self.W_v = nn.Parameter(torch.rand(d_model, d_model))
        
    
    def forward(self, x1, x2):
        batch_size_1, seq_len_1, d_model_1 = x1.shape
        batch_size_2, seq_len_2, d_model_2 = x2.shape
        assert batch_size_1 == batch_size_2, "Inputs x1 and x2 should have same batch size!"
        assert d_model_1 == d_model_2 == self.d_model, "Model dimension mismatch!"
        
        # dim = (batch_size, seq_len, d_model) -> (batch_size, n_heads, seq_len, d_heads)
        q = (x1 @ self.W_q).reshape(batch_size_1, seq_len_1, self.n_heads, self.d_head).transpose(2,1).contiguous()
        k = (x2 @ self.W_k).reshape(batch_size_2, seq_len_2, self.n_heads, self.d_head).transpose(2,1).contiguous()
        v = (x2 @ self.W_v).reshape(batch_size_2, seq_len_2, self.n_heads, self.d_head).transpose(2,1).contiguous()
        
        # dim = (batch_size, n_heads, seq_len_1, seq_len_2)
        attn = q @ k.transpose(-2, -1) * self.scale
        norm_attn = torch.softmax(attn, dim=-1)
        # dim = (batch_size, n_heads, seq_len_1, d_head)
        output = (norm_attn @ v).transpose(2, 1).reshape(batch_size_1, seq_len_1, self.d_model).contiguous()
        return output

In [260]:
d_model = 48
n_heads = 8
ca = CrossAttention(d_model, n_heads)

In [261]:
x1_numpy = np.random.randn(1, 25, 48)
x2_numpy = np.random.randn(1, 20, 48)
x1 = torch.from_numpy(x1_numpy).float()
x2 = torch.from_numpy(x2_numpy).float()

In [262]:
output = ca(x1, x2)

In [263]:
output.size()

torch.Size([1, 25, 48])

In [ ]:
# This is not correct, do better next time
class TranslationModel(nn.Module):
    def __init__(self, n_layers, d_model, n_heads):
        super().__init__()
        self.self_attn_input = nn.ModuleList([
            TransformerBlock(d_model, n_heads)
            for _ in range(n_layers)
        ])
        self.self_attn_target = nn.ModuleList([
            TransformerBlock(d_model, n_heads)
            for _ in range(n_layers)
        ])
        self.cross_attn = CrossAttention(d_model, n_heads)
        
    def forward(self, input_seq, target_seq):
        x1 = target_seq
        x2 = input_seq
        for target_layer in self.self_attn_target:
            x1 = target_layer(x1)

        for input_layer in self.self_attn_input:
            x2 = input_layer(x1)
        
        
        return output_seq

In [264]:
self_attn_block = TransformerBlock(d_model, n_heads)